# Embellishment Eval Results — 2026-03-25 5:37AM

First full embellishment eval run against the expanded 60-question test set (20 per category). Compares four prompt variants using `rrf_elser` as the fixed retrieval strategy.

## Key Findings

- **`original` is a clear outlier** — 11 flagged questions and a delta of 0.88, nearly double any other variant
- **`clear_direct` and `no_adjectives` are essentially tied** — both around 0.42–0.47 delta and 3–4 flagged. The earlier 10-question run had `clear_direct` winning clearly but at 60 questions the difference disappears
- **Lore questions embellish the most regardless of prompt** — source scores around 1.50–1.55, response scores 2.30–2.60 across all variants
- **Item questions embellish the least** — deltas of 0.20–0.60 depending on variant
- **`clear_direct` best reduces lore delta** — from 1.05 (original) to 0.75

In [1]:
import json
from pathlib import Path

RESULTS    = Path('../../evals/generation/embellishment_results_2026-03-25_05-37am.json')
VARIANTS   = ['original', 'clear_direct', 'match_tone', 'no_adjectives']
CATEGORIES = ['gameplay', 'item', 'lore']

data = json.loads(RESULTS.read_text())

## Overall Results

In [2]:
col = 14
print(f"{'':18}", end='')
for v in VARIANTS:
    print(f'  {v:>{col}}', end='')
print()
print('-' * (18 + (col + 2) * len(VARIANTS)))

for field in ['source_score', 'response_score', 'delta']:
    avgs = {v: sum(r[field] for r in data[v]) / len(data[v]) for v in VARIANTS}
    print(f'{field:<18}', end='')
    for v in VARIANTS:
        print(f'  {avgs[v]:>{col}.2f}', end='')
    print()

                          original    clear_direct      match_tone   no_adjectives
----------------------------------------------------------------------------------
source_score                  1.32            1.33            1.30            1.32
response_score                2.20            1.75            1.90            1.78
delta                         0.88            0.42            0.60            0.47


## Per-Category Breakdown

In [3]:
for category in CATEGORIES:
    print(f'\n{category}')
    for field in ['source_score', 'response_score', 'delta']:
        print(f'  {field:<18}', end='')
        for v in VARIANTS:
            cat_results = [r for r in data[v] if r['category'] == category]
            avg = sum(r[field] for r in cat_results) / len(cat_results)
            print(f'  {avg:>12.2f}', end='')
        print()


gameplay
  source_score              1.25          1.25          1.30          1.25
  response_score            2.25          1.55          1.70          1.55
  delta                     1.00          0.30          0.40          0.30

item
  source_score              1.15          1.20          1.10          1.20
  response_score            1.75          1.40          1.45          1.45
  delta                     0.60          0.20          0.35          0.25

lore
  source_score              1.55          1.55          1.50          1.50
  response_score            2.60          2.30          2.55          2.35
  delta                     1.05          0.75          1.05          0.85


## High Embellishment Flags (delta >= 2)

In [4]:
for v in VARIANTS:
    flagged = [r for r in data[v] if r['delta'] >= 2]
    print(f'{v} — {len(flagged)} flagged')
    for r in flagged:
        print(f"  [{r['category']}] {r['question'][:70]}")
        print(f"    source={r['source_score']}  response={r['response_score']}  delta={r['delta']:+d}")
        print(f"    {r['response_reasoning']}")
    print()

original — 11 flagged
  [lore] What is Frostmourne?
    source=1  response=3  delta=+2
    The text blends narrative elements with structured sections that use bold headers and factual lists of the weapon's properties.

  [lore] Who is Sylvanas Windrunner?
    source=2  response=4  delta=+2
    The text flows as continuous prose with clear cause-and-effect storytelling and character arc.

  [lore] Who was Yol'Tithian and what did he do during the War of the Ancients?
    source=1  response=3  delta=+2
    The text uses flowing sentences with causal connections and a clear voice, but remains somewhat structured.

  [lore] Who is Bolten Vanyst and what's his connection to Tyrathan Khort?
    source=2  response=4  delta=+2
    The text flows as continuous prose with clear cause-and-effect relationships and character motivations.

  [gameplay] What's the deal with Iron Qon in Throne of Thunder?
    source=2  response=4  delta=+2
    The text flows smoothly with connecting prose and explana